In [10]:
pip install thop torchinfo

Note: you may need to restart the kernel to use updated packages.


In [11]:
import torch
import time
import os
from thop import profile

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_model_stats(model, input_size=(1, 3, 224, 224), repetitions=100, warmup=10):
    model = model.to(device)
    model.eval()

    dummy_input = torch.randn(input_size).to(device)

    # Params
    params = sum(p.numel() for p in model.parameters()) / 1e6

    # FLOPs
    try:
        flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
        flops = flops / 1e9
    except:
        flops = 0.0

    # Model size
    torch.save(model.state_dict(), "temp.pth")
    size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")

    # Inference time
    timings = []
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy_input)

        if device.type == "cuda":
            starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            for _ in range(repetitions):
                starter.record()
                _ = model(dummy_input)
                ender.record()
                torch.cuda.synchronize()
                timings.append(starter.elapsed_time(ender))
        else:
            for _ in range(repetitions):
                start = time.time()
                _ = model(dummy_input)
                end = time.time()
                timings.append((end - start) * 1000)

    inf_time = sum(timings) / len(timings)

    # GPU memory
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = model(dummy_input)
        mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        mem = 0.0

    return {
        "Params(M)": round(params, 2),
        "FLOPs(G)": round(flops, 2),
        "Size(MB)": round(size_mb, 2),
        "Inf Time(ms)": round(inf_time, 2),
        "GPU Mem(GB)": round(mem, 2)
    }

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class MSW_EnsNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # EfficientNet-B4
        eff = models.efficientnet_b4(weights=None)
        self.eff_features = eff.features
        self.eff_pool = nn.AdaptiveAvgPool2d(1)
        self.eff_out = 1792

        # InceptionV3 (correct usage)
        inc = models.inception_v3(weights=None, aux_logits=False)
        self.inc_backbone = inc
        self.inc_backbone.fc = nn.Identity()  # remove classifier
        self.inc_out = 2048

        # Learnable weights
        self.weights = nn.Parameter(torch.ones(2))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(self.eff_out + self.inc_out, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # EfficientNet branch (224 input)
        f1 = self.eff_features(x)
        f1 = self.eff_pool(f1)
        f1 = torch.flatten(f1, 1)

        # Inception branch (RESIZE to 299)
        x_inc = F.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)
        f2 = self.inc_backbone(x_inc)

        # Fusion weights
        w = torch.softmax(self.weights, dim=0)
        f1 = w[0] * f1
        f2 = w[1] * f2

        fused = torch.cat([f1, f2], dim=1)
        return self.classifier(fused)

In [13]:
model = MSW_EnsNet(num_classes=2)

stats = compute_model_stats(model, (1, 3, 224, 224))

print(stats)

{'Params(M)': 41.3, 'FLOPs(G)': 4.43, 'Size(MB)': 158.79, 'Inf Time(ms)': 26.05, 'GPU Mem(GB)': 0.5}
